# Detecção de anomalias
* modelos não supervisionados para detectar anomalias.

## Pacotes

In [0]:
%run
/CMSP/Utils/configuration

Python interpreter will be restarted.
Collecting imblearn
 Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
Collecting imbalanced-learn
 Using cached imbalanced_learn-0.9.1-py3-none-any.whl (199 kB)
Requirement already satisfied: joblib>=1.0.0 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.0.1)
Requirement already satisfied: threadpoolctl>=2.0.0 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (2.1.0)
Collecting scikit-learn>=1.1.0
 Using cached scikit_learn-1.1.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (31.2 MB)
Requirement already satisfied: scipy>=1.3.2 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.6.2)
Requirement already satisfied: numpy>=1.17.3 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.19.2)
Installing collected packages: scikit-learn, imbalanced-learn, imblearn
 Attempting uninstall: scikit-learn
 Found existing installation: scikit-learn 0.24.1
 Not uninstalling scikit-learn at /databricks/python3/lib/python3.8/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-642e6e8a-648d-49e3-8b0f-5434863f73c8
 Can't uninstall 'scikit-learn'. No files were found to uninstall.
Successfully installed imbalanced-learn-0.9.1 imblearn-0.0 scikit-learn-1.1.1
WARNING: You are using pip version 21.0.1; however, version 22.1.2 is available.
You should consider upgrading via the '/local_disk0/.ephemeral_nfs/envs/pythonEnv-642e6e8a-648d-49e3-8b0f-5434863f73c8/bin/python -m pip install --upgrade pip' command.
Python interpreter will be restarted.

Out[2]: DataFrame[]

# Methods

* As funções abaixo, necessitam ser usadas em conjunto.

## ident_outliers_isolation_forest

### Técnica Isolation Forest
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html?highlight=isolationforest#sklearn.ensemble.IsolationForest

* Isola as observações randomicamente selecionando as variáveis, e depois seleciona randomicamente um valor de corte entre o minimo e o maximo da variável selecionada.
* é representado por uma estrutura de arvore, o número de divisões necessárias para isolar uma amostra é equivalente ao comprimento do caminho do nó raiz até o nó final.
* algoritmo para detecção de anomalias, ele retira tanto os outliers variados como também os multivariados.
* identifica as anomalias partindo do pressuposto que uma arvore de decisão encontrará as anomalias por aquela que a análise percorrer o menor caminho;

### Função

In [0]:
def ident_outliers_isolation_forest(dados, target, filtro = "filtro", coluna = "coluna", n_estimators=100, max_samples='auto', contamination='auto', max_features=1.0, bootstrap=False, n_jobs=None, random_state=None, verbose=0, warm_start=False):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/3749511783433397/command/3749511783433409
  '''
  dado = recebe um dataset completo com variáveis preditoras e target - o dataset necessita estar em formato Pandas.
  target = nome da coluna que há a variável target
  retorno = dataset com novas colunas agregadas, contendo 1 e -1. os registros que possuirem -1 em qualquer coluna de identificação são consideradas outliers.
  '''
  dados = dados.toPandas() #transforma os dados em pandas
  colunas = dados.columns #armazena as colunas em uma lista
  tg_list = dados[target].unique() #* verificar as classes distintas
  name_col = list(dados.drop(target, axis=1)) #* selecionar a coluna
  df_retorno = []
  var_outlier = []
  for values, tg in enumerate(tg_list):
    subset = dados[dados[target] == tg] # quebar o dataset por classe em outros df
    for variavel in name_col:
      clf = IsolationForest(n_estimators=n_estimators, max_samples=max_samples, contamination=contamination, max_features=max_features, bootstrap=bootstrap, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start) #estimador
      X = np.array(subset[variavel]) #transforma as variaveis target em array
      X = np.reshape(X, (len(X), 1)) #dá um reshape nos dados para que sejam armazenados linhas(registros ao invez de coluna)
      clf.fit(X)
      predict = clf.predict(X)
      subset[f'{variavel}_outlier'] = predict #armazena a informação para uma nova coluna
      filtro_coluna = f'{variavel}_outlier' #filtra a coluna trabalhada no loop
      subset = subset[subset[filtro_coluna] > 0] #filtra os dados preditos (-1) da coluna trabalhada no loop
    df_retorno.append(subset) #junta as classes
  retorno = pd.concat(df_retorno) #armazena os dados numa variável
  retorno = retorno[colunas] # filtra as colunas do dataset de entrada
  retorno = spark.createDataFrame(retorno) #transformando no dataframe spark
  return retorno